# LLM-Based Android App Permission Multilabel Classification Pipeline

This notebook implements a modular pipeline for **multilabel classification** of Android app permissions using different Large Language Models (LLMs) with **few-shot learning**.

## Key Features:
- **Few-shot Learning**: Uses example demonstrations to guide the model
- **Multilabel Classification**: Predicts multiple permissions per app (not just binary)
- **Modular Design**: Easy to add new models or modify existing ones
- **Multiple LLM Support**: Llama 2, Mistral, FLAN-T5, Phi-3, and more

## Task:
Given an app description, predict which permissions from a predefined list the app will request.

## Question Format:
"Based on this description, out of the list of permissions what permission do you think this app will request?"


## 1. Setup and Imports


In [ ]:
# Install required packages
%pip install transformers torch pandas numpy scikit-learn matplotlib seaborn accelerate bitsandbytes requests openpyxl


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 50.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM,
    pipeline, BitsAndBytesConfig
)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, hamming_loss, jaccard_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json
from typing import Dict, List, Tuple, Any, Optional
import time
from tqdm import tqdm
import os
from datetime import datetime
import re

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)


Using device: cuda


## 2. Configuration and Constants


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Configuration
CONFIG = {
    'data_path': '/content/drive/MyDrive/data/apps_with_permissions_cleaned_with_names_and_NEW_permissions_TOP30.csv',
    'target_permissions': [
        'Other : full network access',
        'Other : view network connections',
        'Other : prevent device from sleeping',
        'Photos/Media/Files : read the contents of your USB storage',
        'Storage : read the contents of your USB storage',
        'Uncategorized : receive data from Internet',
        'Photos/Media/Files : modify or delete the contents of your USB storage',
        'Storage : modify or delete the contents of your USB storage',
        'Other : control vibration',
        'Other : run at startup',
        'Camera : take pictures and videos',
        'Wi-Fi connection information : view Wi-Fi connections',
        'Location : precise location (GPS and network-based)',
        'Location : approximate location (network-based)',
        'Other : pair with Bluetooth devices',
        'Microphone : record audio',
        'Other : change your audio settings',
        'Other : access Bluetooth settings',
        'Calendar : read calendar events plus confidential information',
        "Calendar : add or modify calendar events and send email to guests without owners' knowledge",
        'Other : Google Play license check',
        'Contacts : read your contacts',
        'Device ID & call information : read phone status and identity',
        'Phone : read phone status and identity',
        'Other : draw over other apps',
        'Other : read Google service configuration',
        'Phone : directly call phone numbers',
        'Contacts : find accounts on the device',
        'Identity : find accounts on the device',
        'Other : connect and disconnect from Wi-Fi'
    ],
    'llm_models': {
        'llama2': 'meta-llama/Llama-2-7b-chat-hf', # 'TheBloke/Llama-2-7B-Chat-GGUF'
        'mistral': 'unsloth/mistral-7b-instruct-v0.3-bnb-4bit', # 'mistralai/Mistral-7B-Instruct-v0.3'
        'flan-t5': 'google/flan-t5-large',
        'phi-3': 'leliuga/Phi-3-mini-4k-instruct-bnb-4bit', # 'microsoft/Phi-3-mini-4k-instruct'
    },
    'max_length': 512,
    'batch_size': 8,
    'test_size': 0.2,
    'random_state': 42,
    'few_shot_examples': 5,  # Number of few-shot examples to include
    'zero_shot_prompt_mode': 'enhanced',  # 'enhanced' for detailed zero-shot instructions, 'simple' for basic
    'results_dir': '/content/drive/MyDrive/LLMs/multilabel_classification/results/'
}

print(f"Target permissions: {len(CONFIG['target_permissions'])}")
print(f"LLM models: {list(CONFIG['llm_models'].keys())}")
print(f"Few-shot examples: {CONFIG['few_shot_examples']}")


Target permissions: 30
LLM models: ['llama2', 'mistral', 'flan-t5', 'phi-3']
Few-shot examples: 5


In [ ]:
class DataPreprocessor:
    """
    Handles data loading, cleaning, and preprocessing for multilabel permission classification.
    """

    def __init__(self, config: Dict[str, Any]):
        self.config = config
        self.data = None
        self.processed_data = None

    def load_data(self) -> pd.DataFrame:
        """
        Load the CSV data and perform initial cleaning.
        """
        print("Loading data...")
        self.data = pd.read_csv(self.config['data_path'])

        print(f"Data shape: {self.data.shape}")
        print(f"Columns: {len(self.data.columns)}")

        # Check if required columns exist
        required_cols = ['description'] + self.config['target_permissions']
        missing_cols = [col for col in required_cols if col not in self.data.columns]

        if missing_cols:
            print(f"Warning: Missing columns: {missing_cols}")
            # Filter to only include available permissions
            available_permissions = [col for col in self.config['target_permissions'] if col in self.data.columns]
            self.config['target_permissions'] = available_permissions
            print(f"Using available permissions: {len(available_permissions)}")

        return self.data

    def clean_text(self, text: str) -> str:
        """
        Clean and preprocess text data.
        """
        if pd.isna(text) or text == '':
            return ''

        # Remove HTML tags
        text = re.sub(r'<[^>]+>', '', str(text))
        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text)
        # Remove special characters but keep basic punctuation
        text = re.sub(r'[^\w\s.,!?()-]', '', text)

        return text.strip()

    def preprocess_data(self) -> pd.DataFrame:
        """
        Preprocess the data for multilabel classification.
        """
        if self.data is None:
            self.load_data()

        print("Preprocessing data...")

        # Clean descriptions
        self.data['description'] = self.data['description'].apply(self.clean_text)

        # Remove rows with empty descriptions
        initial_count = len(self.data)
        self.data = self.data[self.data['description'].str.len() > 10]
        print(f"Removed {initial_count - len(self.data)} rows with empty/short descriptions")

        # Convert permission columns to binary (0/1)
        for perm in self.config['target_permissions']:
            if perm in self.data.columns:
                self.data[perm] = self.data[perm].astype(float).fillna(0)
                self.data[perm] = (self.data[perm] > 0).astype(int)

        self.processed_data = self.data.copy()

        print(f"Final data shape: {self.processed_data.shape}")
        return self.processed_data

    def split_data(self, test_size: float = None) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """
        Split data into train and test sets.
        """
        if self.processed_data is None:
            self.preprocess_data()

        test_size = test_size or self.config['test_size']

        train_data, test_data = train_test_split(
            self.processed_data,
            test_size=test_size,
            random_state=self.config['random_state']
        )

        print(f"Train set: {len(train_data)} samples")
        print(f"Test set: {len(test_data)} samples")

        return train_data, test_data

    def get_few_shot_examples(self, train_data: pd.DataFrame, n_examples: int = 3) -> List[Dict[str, Any]]:
        """
        Get diverse few-shot examples from training data.
        Improved version that selects examples with varying numbers of permissions.
        Returns a list of examples with description and list of permissions.
        """
        examples = []
        seen_permission_counts = set()

        # First, collect all valid examples
        all_examples = []
        for idx in range(len(train_data)):
            row = train_data.iloc[idx]

            # Get permissions for this app
            permissions = []
            for perm in self.config['target_permissions']:
                if perm in row.index and row[perm] == 1:
                    permissions.append(perm)

            # Only include examples with at least 1 permission
            if len(permissions) > 0:
                all_examples.append({
                    'description': row['description'],
                    'permissions': permissions,
                    'count': len(permissions)
                })

        # Sort by permission count to get diverse examples
        all_examples.sort(key=lambda x: x['count'])

        # Select diverse examples: one with few permissions, one with medium, one with many
        if len(all_examples) >= n_examples:
            # Get examples from different parts of the distribution
            indices = [
                0,  # Few permissions
                len(all_examples) // 2,  # Medium permissions
                len(all_examples) - 1  # Many permissions
            ]

            for idx in indices[:n_examples]:
                if idx < len(all_examples):
                    examples.append({
                        'description': all_examples[idx]['description'],
                        'permissions': all_examples[idx]['permissions']
                    })
        else:
            # If not enough examples, just take what we have
            for ex in all_examples[:n_examples]:
                examples.append({
                    'description': ex['description'],
                    'permissions': ex['permissions']
                })

        # Shuffle to avoid bias
        np.random.shuffle(examples)

        return examples[:n_examples]

# Initialize preprocessor
preprocessor = DataPreprocessor(CONFIG)
data = preprocessor.load_data()
processed_data = preprocessor.preprocess_data()


Loading data...
Data shape: (5348, 36)
Columns: 36
Preprocessing data...
Removed 0 rows with empty/short descriptions
Final data shape: (5348, 36)


In [ ]:
class LLMMultilabelClassifier:
    """
    Base class for LLM-based multilabel permission classification with few-shot learning.
    """

    def __init__(self, model_name: str, config: Dict[str, Any]):
        self.model_name = model_name
        self.config = config
        self.model = None
        self.tokenizer = None
        self.pipeline = None
        self.device = device

    def load_model(self):
        """Load the model and tokenizer. To be implemented by subclasses."""
        raise NotImplementedError

    def create_prompt(self, description: str, few_shot_examples: List[Dict[str, Any]] = None) -> str:
        """
        Create a prompt for multilabel classification with few-shot examples or zero-shot.

        Args:
            description: App description to classify
            few_shot_examples: List of example dicts with 'description' and 'permissions' keys
                             If None, uses zero-shot classification with enhanced instructions
        """
        # Build permission list string
        permissions_list = "\n".join([f"{i+1}. {perm}" for i, perm in enumerate(self.config['target_permissions'])])

        # Build few-shot examples section
        examples_text = ""
        zero_shot_instructions = ""

        if few_shot_examples:
            examples_text = "\n\nHere are some examples:\n\n"
            for i, example in enumerate(few_shot_examples, 1):
                example_perms = ", ".join(example['permissions'])
                examples_text += f"Example {i}:\n"
                examples_text += f"Description: {example['description'][:200]}...\n"
                examples_text += f"Permissions: {example_perms}\n\n"
        else:
            # Zero-shot: Add enhanced instructions if enabled
            prompt_mode = self.config.get('zero_shot_prompt_mode', 'enhanced')
            if prompt_mode == 'enhanced':
                zero_shot_instructions = """
Instructions for zero-shot classification:
1. Carefully read the app description to understand its functionality
2. Identify ALL permissions the app will need based on its features
3. Consider both explicit features (e.g., "takes photos" → Camera permission) and implicit needs (e.g., "social media app" → Network access)
4. Output your answer in the format: "Permissions: [comma-separated numbers]" (e.g., "Permissions: 1, 5, 12, 23")
5. Only include permissions that are clearly necessary for the app's described functionality
6. If no permissions are needed, output: "Permissions: none"

"""

        prompt = f"""Task: Based on the app description, predict which permissions from the list below this app will request.

{examples_text}{zero_shot_instructions}
Now, analyze the following app description:

App Description: {description}

List of possible permissions:
{permissions_list}

Question: Based on this description, out of the list of permissions what permission do you think this app will request?

Please list the permission names (or numbers) that this app will request. If the app requests multiple permissions, list them all separated by commas."""

        return prompt.strip()

    def parse_response(self, response_text: str) -> List[int]:
        """
        Parse the model's response to extract predicted permissions.
        Returns a binary vector (list of 0s and 1s) indicating which permissions were predicted.
        """
        # Strip whitespace and remove surrounding quotes if present
        response_text = response_text.strip().strip('"').strip("'")
        original_response = response_text  # Keep original for debugging

        # Initialize prediction vector (all zeros)
        predictions = [0] * len(self.config['target_permissions'])

        # First, try to extract structured format: "Permissions: [numbers]" or "Permissions: 11, 4, 8"
        # Improved regex to handle both "Permissions: 11, 4, 8" and "Permissions: [11, 4, 8]"
        permissions_match = re.search(r'Permissions?:\s*(?:\[)?([^\]]+?)(?:\])?', response_text, re.IGNORECASE)
        if permissions_match:
            numbers_str = permissions_match.group(1).strip()
            # Extract numbers from the matched string
            numbers = re.findall(r'(\d+)', numbers_str)
            for num_str in numbers:
                try:
                    num = int(num_str) - 1  # Convert to 0-based index
                    if 0 <= num < len(predictions):
                        predictions[num] = 1
                except ValueError:
                    continue
            # If we found permissions in structured format, return early
            if sum(predictions) > 0:
                return predictions

        # Try to find "Permissions: none" or similar
        if re.search(r'Permissions?:\s*(none|no permissions)', response_text, re.IGNORECASE):
            return predictions  # All zeros

        # Try to find permission names or numbers in the response
        response_lower = response_text.lower()
        for i, perm in enumerate(self.config['target_permissions']):
            # Check if permission name appears in response (case-insensitive)
            perm_lower = perm.lower()
            # Use word boundaries to avoid partial matches
            if re.search(r'' + re.escape(perm_lower) + r'', response_lower):
                predictions[i] = 1
            # Also check for permission number (e.g., "1", "2", etc.)
            if f"{i+1}." in response_text or f"permission {i+1}" in response_lower:
                predictions[i] = 1

        # If no permissions found, try to extract all numbers
        if sum(predictions) == 0:
            numbers = re.findall(r'(\d+)', response_text)
            for num_str in numbers:
                try:
                    num = int(num_str) - 1  # Convert to 0-based index
                    if 0 <= num < len(predictions):
                        predictions[num] = 1
                except ValueError:
                    continue

        return predictions
    def classify_permissions(self, description: str, few_shot_examples: List[Dict[str, Any]] = None) -> Tuple[List[int], str]:
        """
        Classify which permissions an app requests.
        Returns a tuple of (binary vector indicating predicted permissions, raw response text).
        """
        if self.pipeline is None:
            self.load_model()

        prompt = self.create_prompt(description, few_shot_examples)

        try:
            response = self.pipeline(prompt, max_length=512, do_sample=False)
            if isinstance(response, list):
                response_text = response[0]['generated_text']
            else:
                response_text = response['generated_text']

            # Extract permissions from response
            predictions = self.parse_response(response_text)
            return predictions, response_text

        except Exception as e:
            print(f"Error in classification: {e}")
            return [0] * len(self.config['target_permissions']), ""

    def batch_classify(self, descriptions: List[str], few_shot_examples: List[Dict[str, Any]] = None) -> Tuple[List[List[int]], List[str]]:
        """Classify multiple descriptions. Returns (predictions, raw_responses)."""
        predictions = []
        raw_responses = []

        for desc in tqdm(descriptions, desc=f"Classifying with {self.model_name}..."):
            pred, raw_response = self.classify_permissions(desc, few_shot_examples)
            predictions.append(pred)
            raw_responses.append(raw_response)

        return predictions, raw_responses


In [ ]:
class Llama2MultilabelClassifier(LLMMultilabelClassifier):
    """Llama 2 based multilabel classifier."""

    def load_model(self):
        print(f"Loading {self.model_name} (Llama 2 via HF + bnb 4-bit)...")

        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16
        )

        import os
        hf_token = os.environ.get('HF_TOKEN')
        model_id = self.config['llm_models']['llama2']

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id,
            token=hf_token,
            trust_remote_code=True
        )
        if self.tokenizer.pad_token is None and hasattr(self.tokenizer, 'eos_token'):
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            token=hf_token,
            quantization_config=quantization_config,
            device_map="auto",
            trust_remote_code=True
        )

        self.pipeline = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            device_map="auto"
        )

        print(f"{self.model_name} loaded successfully!")

    def create_prompt(self, description: str, few_shot_examples: List[Dict[str, Any]] = None) -> str:
        """Override to use Llama 2 chat template format with improved prompt."""
        # Build permission list string
        permissions_list = "".join([f"{i+1}. {perm}" for i, perm in enumerate(self.config['target_permissions'])])

        # Build few-shot examples section with explicit output format
        examples_text = ""
        zero_shot_instructions = ""

        if few_shot_examples:
            # Few-shot mode: include examples
            examples_text = "\n\nHere are some examples showing the expected format:\n\n"
            for i, example in enumerate(few_shot_examples, 1):
                # Get permission numbers for this example
                perm_numbers = []
                for perm in example['permissions']:
                    try:
                        idx = self.config['target_permissions'].index(perm)
                        perm_numbers.append(str(idx + 1))
                    except ValueError:
                        continue

                example_perms_str = ", ".join(example['permissions'][:3])
                if len(example['permissions']) > 3:
                    example_perms_str += f" (and {len(example['permissions']) - 3} more)"

                examples_text += f"Example {i}:\n"
                examples_text += f"Description: {example['description'][:250]}...\n"
                examples_text += f"Required Permissions: {', '.join(perm_numbers)}\n"
                examples_text += f"Permission Names: {example_perms_str}\n\n"
        else:
            # Zero-shot mode: add enhanced instructions
            zero_shot_instructions = """
IMPORTANT GUIDELINES FOR ZERO-SHOT CLASSIFICATION:
- Analyze the app description systematically
- Consider what features the app needs (camera, location, storage, network, etc.)
- Match each feature to the corresponding permission from the list
- Think about what permissions are typically required for similar apps
- Be thorough but accurate - only include permissions that are clearly needed
- Multiple permissions can be required for a single app

"""

        prompt_text = f"""You are an expert Android security analyst. Your task is to identify which permissions an Android app will request based on its description.

{zero_shot_instructions}{examples_text}Now analyze this app description:

App Description: {description}

Available Permissions (choose from this list):
{permissions_list}

Instructions:
1. Read the app description carefully
2. Identify ALL permissions the app will need based on its functionality
3. Output your answer in this EXACT format: "Permissions: [comma-separated numbers]"
   For example: "Permissions: 1, 5, 12, 23"
4. Only include permissions that are clearly needed for the app's described features
5. If no permissions are needed, output: "Permissions: none"

Question: Based on this description, out of the list of permissions what permission do you think this app will request?

Answer (format: "Permissions: [numbers]"):

Do not reply anything other that the Answer format. NO EXPLAINATION.

"""

        # Apply Llama 2 chat template
        if hasattr(self.tokenizer, 'apply_chat_template') and hasattr(self.tokenizer, 'chat_template') and self.tokenizer.chat_template:
            messages = [{"role": "user", "content": prompt_text}]
            prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        else:
            # Fallback to Llama 2 format
            prompt = f"<s>[INST] {prompt_text} [/INST]"

        return prompt
    def classify_permissions(self, description: str, few_shot_examples: List[Dict[str, Any]] = None) -> Tuple[List[int], str]:
        """Override with Llama 2 specific settings."""
        if self.pipeline is None:
            self.load_model()

        prompt = self.create_prompt(description, few_shot_examples)

        try:
            response = self.pipeline(
                prompt,
                max_new_tokens=200,
                do_sample=False,
                temperature=0.0,
                truncation=True,
                return_full_text=False
            )

            if isinstance(response, list):
                response_text = response[0].get('generated_text', '')
            else:
                response_text = response.get('generated_text', '')

            predictions = self.parse_response(response_text)
            return predictions, response_text

        except Exception as e:
            print(f"Error in classification: {e}")
            return [0] * len(self.config['target_permissions']), ""


In [ ]:
class MistralMultilabelClassifier(LLMMultilabelClassifier):
    """Mistral based multilabel classifier."""

    def load_model(self):
        print(f"Loading {self.model_name}...")

        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4"
        )

        self.tokenizer = AutoTokenizer.from_pretrained(
            self.config['llm_models']['mistral'],
            trust_remote_code=True
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            self.config['llm_models']['mistral'],
            quantization_config=quantization_config,
            device_map="auto",
            trust_remote_code=True
        )

        self.pipeline = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            device_map="auto"
        )

        print(f"{self.model_name} loaded successfully!")

    def create_prompt(self, description: str, few_shot_examples: List[Dict[str, Any]] = None) -> str:
        """Override to use Mistral's chat template format with improved prompt."""
        # Build permission list string
        permissions_list = "\n".join([f"{i+1}. {perm}" for i, perm in enumerate(self.config['target_permissions'])])

        # Build few-shot examples section with explicit output format
        examples_text = ""
        zero_shot_instructions = ""

        if few_shot_examples:
            # Few-shot mode: include examples
            examples_text = "\n\nHere are some examples showing the expected format:\n\n"
            for i, example in enumerate(few_shot_examples, 1):
                # Get permission numbers for this example
                perm_numbers = []
                for perm in example['permissions']:
                    try:
                        idx = self.config['target_permissions'].index(perm)
                        perm_numbers.append(str(idx + 1))
                    except ValueError:
                        continue

                example_perms_str = ", ".join(example['permissions'][:3])
                if len(example['permissions']) > 3:
                    example_perms_str += f" (and {len(example['permissions']) - 3} more)"

                examples_text += f"Example {i}:\n"
                examples_text += f"Description: {example['description'][:250]}...\n"
                examples_text += f"Required Permissions: {', '.join(perm_numbers)}\n"
                examples_text += f"Permission Names: {example_perms_str}\n\n"
        else:
            # Zero-shot mode: add enhanced instructions
            zero_shot_instructions = """
IMPORTANT GUIDELINES FOR ZERO-SHOT CLASSIFICATION:
- Analyze the app description systematically
- Consider what features the app needs (camera, location, storage, network, etc.)
- Match each feature to the corresponding permission from the list
- Think about what permissions are typically required for similar apps
- Be thorough but accurate - only include permissions that are clearly needed
- Multiple permissions can be required for a single app

"""

        prompt_text = f"""You are an expert Android security analyst. Your task is to identify which permissions an Android app will request based on its description.

{zero_shot_instructions}{examples_text}Now analyze this app description:

App Description: {description}

Available Permissions (choose from this list):
{permissions_list}

Instructions:
1. Read the app description carefully
2. Identify ALL permissions the app will need based on its functionality
3. Output your answer in this EXACT format: "Permissions: [comma-separated numbers]"
   For example: "Permissions: 1, 5, 12, 23"
4. Only include permissions that are clearly needed for the app's described features
5. If no permissions are needed, output: "Permissions: none"

Question: Based on this description, out of the list of permissions what permission do you think this app will request?

Answer (format: "Permissions: [numbers]"):"""

        # Apply Mistral chat template if available
        if hasattr(self.tokenizer, 'apply_chat_template') and hasattr(self.tokenizer, 'chat_template') and self.tokenizer.chat_template:
            messages = [{"role": "user", "content": prompt_text}]
            prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        else:
            prompt = f"[INST] {prompt_text} [/INST]"

        return prompt

    def classify_permissions(self, description: str, few_shot_examples: List[Dict[str, Any]] = None) -> Tuple[List[int], str]:
        """Override with Mistral specific settings."""
        if self.pipeline is None:
            self.load_model()

        prompt = self.create_prompt(description, few_shot_examples)

        try:
            response = self.pipeline(
                prompt,
                max_new_tokens=200,
                do_sample=False,
                temperature=0.0,
                truncation=True,
                return_full_text=False
            )

            if isinstance(response, list):
                response_text = response[0].get('generated_text', '')
            else:
                response_text = response.get('generated_text', '')

            predictions = self.parse_response(response_text)
            return predictions, response_text

        except Exception as e:
            print(f"Error in Mistral classification: {e}")
            return [0] * len(self.config['target_permissions']), ""


In [ ]:
class FlanT5MultilabelClassifier(LLMMultilabelClassifier):
    """FLAN-T5 based multilabel classifier."""

    def load_model(self):
        print(f"Loading {self.model_name}...")

        self.tokenizer = AutoTokenizer.from_pretrained(
            self.config['llm_models']['flan-t5']
        )

        self.model = AutoModelForSeq2SeqLM.from_pretrained(
            self.config['llm_models']['flan-t5'],
            device_map="auto"
        )

        self.pipeline = pipeline(
            "text2text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            device_map="auto"
        )

        print(f"{self.model_name} loaded successfully!")

    def classify_permissions(self, description: str, few_shot_examples: List[Dict[str, Any]] = None) -> Tuple[List[int], str]:
        """Override with FLAN-T5 specific settings."""
        if self.model is None or self.tokenizer is None:
            self.load_model()

        prompt = self.create_prompt(description, few_shot_examples)

        try:
            # Tokenize the input prompt
            inputs = self.tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=512
            ).to(self.model.device)

            # Generate response using model.generate() directly
            # This ensures we only get the generated tokens, not the input prompt
            with torch.no_grad():
                generated_ids = self.model.generate(
                    inputs.input_ids,
                    max_new_tokens=200,
                    do_sample=False,
                    temperature=1.0,
                    num_beams=1,
                    pad_token_id=self.tokenizer.pad_token_id if self.tokenizer.pad_token_id is not None else self.tokenizer.eos_token_id,
                    eos_token_id=self.tokenizer.eos_token_id
                )

            # Decode only the generated tokens (skip the input prompt)
            # For seq2seq models, we need to decode only the new tokens
            input_length = inputs.input_ids.shape[1]
            generated_tokens = generated_ids[0][input_length:]
            response_text = self.tokenizer.decode(
                generated_tokens,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True
            )

            predictions = self.parse_response(response_text)
            return predictions, response_text

        except Exception as e:
            print(f"Error in FLAN-T5 classification: {e}")
            import traceback
            traceback.print_exc()
            return [0] * len(self.config['target_permissions']), ""


class Phi3MultilabelClassifier(LLMMultilabelClassifier):
    """Phi-3 based multilabel classifier."""

    def load_model(self):
        print(f"Loading {self.model_name}...")

        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4"
        )

        self.tokenizer = AutoTokenizer.from_pretrained(
            self.config['llm_models']['phi-3'],
            trust_remote_code=True
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            self.config['llm_models']['phi-3'],
            quantization_config=quantization_config,
            device_map="auto",
            trust_remote_code=True
        )

        self.pipeline = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            device_map="auto"
        )

        print(f"{self.model_name} loaded successfully!")

    def classify_permissions(self, description: str, few_shot_examples: List[Dict[str, Any]] = None) -> Tuple[List[int], str]:
        """Override with Phi-3 specific settings."""
        if self.pipeline is None:
            self.load_model()

        prompt = self.create_prompt(description, few_shot_examples)

        try:
            response = self.pipeline(
                prompt,
                max_new_tokens=200,
                do_sample=False,
                temperature=0.0,
                truncation=True,
                return_full_text=False
            )

            if isinstance(response, list):
                response_text = response[0].get('generated_text', '')
            else:
                response_text = response.get('generated_text', '')

            predictions = self.parse_response(response_text)
            return predictions, response_text

        except Exception as e:
            print(f"Error in Phi-3 classification: {e}")
            return [0] * len(self.config['target_permissions']), ""


In [ ]:
class MultilabelEvaluationMetrics:
    """
    Handles evaluation metrics for multilabel classification.
    """

    @staticmethod
    def calculate_metrics(y_true: List[List[int]], y_pred: List[List[int]]) -> Dict[str, float]:
        """
        Calculate multilabel classification metrics.

        Args:
            y_true: List of true label vectors (each is a list of 0s and 1s)
            y_pred: List of predicted label vectors (each is a list of 0s and 1s)
        """
        y_true_array = np.array(y_true)
        y_pred_array = np.array(y_pred)

        metrics = {
            'hamming_loss': hamming_loss(y_true_array, y_pred_array),
            'exact_match_ratio': accuracy_score(y_true_array, y_pred_array),
            'jaccard_score': jaccard_score(y_true_array, y_pred_array, average='macro', zero_division=0),
            'precision_macro': precision_score(y_true_array, y_pred_array, average='macro', zero_division=0),
            'recall_macro': recall_score(y_true_array, y_pred_array, average='macro', zero_division=0),
            'f1_macro': f1_score(y_true_array, y_pred_array, average='macro', zero_division=0),
            'precision_micro': precision_score(y_true_array, y_pred_array, average='micro', zero_division=0),
            'recall_micro': recall_score(y_true_array, y_pred_array, average='micro', zero_division=0),
            'f1_micro': f1_score(y_true_array, y_pred_array, average='micro', zero_division=0),
        }

        return metrics

    @staticmethod
    def calculate_per_label_metrics(y_true: List[List[int]], y_pred: List[List[int]],
                                    permission_names: List[str]) -> pd.DataFrame:
        """Calculate metrics for each permission label."""
        y_true_array = np.array(y_true)
        y_pred_array = np.array(y_pred)

        results = []
        for i, perm_name in enumerate(permission_names):
            y_true_label = y_true_array[:, i]
            y_pred_label = y_pred_array[:, i]

            results.append({
                'Permission': perm_name,
                'Precision': precision_score(y_true_label, y_pred_label, zero_division=0),
                'Recall': recall_score(y_true_label, y_pred_label, zero_division=0),
                'F1-Score': f1_score(y_true_label, y_pred_label, zero_division=0),
                'Support': np.sum(y_true_label)
            })

        return pd.DataFrame(results)

    @staticmethod
    def plot_metrics_comparison(results: Dict[str, Dict[str, float]]):
        """Plot comparison of metrics across models."""
        models = list(results.keys())
        metrics = ['hamming_loss', 'exact_match_ratio', 'f1_macro', 'f1_micro']

        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        axes = axes.flatten()

        for i, metric in enumerate(metrics):
            ax = axes[i]
            values = [results[model][metric] for model in models]

            bars = ax.bar(models, values, color=['skyblue', 'lightgreen', 'lightcoral', 'gold'][:len(models)])
            ax.set_title(f'{metric.replace("_", " ").title()}')
            ax.set_ylabel('Score')
            ax.set_ylim(0, 1 if metric != 'hamming_loss' else max(values) * 1.2)

            # Add value labels
            for bar, value in zip(bars, values):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                       f'{value:.3f}', ha='center', va='bottom', fontweight='bold')

        plt.tight_layout()
        plt.show()


In [ ]:
class MultilabelClassificationPipeline:
    """
    Main pipeline for running multilabel permission classification experiments.
    """

    def __init__(self, config: Dict[str, Any]):
        self.config = config
        self.preprocessor = DataPreprocessor(config)
        self.evaluator = MultilabelEvaluationMetrics()
        self.results = {}

        # Create results directory
        os.makedirs(config['results_dir'], exist_ok=True)

    def get_classifier(self, model_name: str):
        """Get the appropriate classifier instance based on model name."""
        if model_name.lower() == 'llama2':
            return Llama2MultilabelClassifier(model_name, self.config)
        elif model_name.lower() == 'mistral':
            return MistralMultilabelClassifier(model_name, self.config)
        elif model_name.lower() == 'flan-t5':
            return FlanT5MultilabelClassifier(model_name, self.config)
        elif model_name.lower() == 'phi-3':
            return Phi3MultilabelClassifier(model_name, self.config)
        else:
            raise ValueError(f"Unknown model: {model_name}")

    def run_experiment(self, model_name: str, sample_size: int = None,
                      use_few_shot: bool = True) -> Dict[str, Any]:
        """
        Run multilabel classification experiment for a specific model.

        Args:
            model_name: Name of the LLM model to use
            sample_size: Number of samples to use (None for all data)
            use_few_shot: Whether to use few-shot learning

        Returns:
            Dictionary with results
        """
        print(f"\n{'='*60}")
        print(f"Running multilabel classification experiment with {model_name}")
        print(f"Few-shot learning: {use_few_shot}")
        print(f"{'='*60}")

        # Initialize classifier
        classifier = self.get_classifier(model_name)

        # Load and preprocess data
        data = self.preprocessor.preprocess_data()

        # Sample data if specified
        if sample_size and sample_size < len(data):
            data = data.sample(n=sample_size, random_state=self.config['random_state'])
            self.preprocessor.processed_data = data
            print(f"Using sample of {sample_size} records")

        # Split data
        train_data, test_data = self.preprocessor.split_data()

        # Get few-shot examples from training data
        few_shot_examples = None
        if use_few_shot:
            few_shot_examples = self.preprocessor.get_few_shot_examples(
                train_data,
                n_examples=self.config['few_shot_examples']
            )
            print(f"\nFew-shot examples ({len(few_shot_examples)}):")
            for i, ex in enumerate(few_shot_examples, 1):
                print(f"  Example {i}: {len(ex['permissions'])} permissions")

        # Get test descriptions and true labels
        test_descriptions = test_data['description'].tolist()

        # Build true label vectors
        y_true = []
        for _, row in test_data.iterrows():
            true_labels = [int(row[perm]) for perm in self.config['target_permissions']]
            y_true.append(true_labels)

        print(f"\nClassifying {len(test_descriptions)} test samples...")

        # Get predictions and raw responses
        y_pred, raw_responses = classifier.batch_classify(test_descriptions, few_shot_examples)

        # Calculate metrics
        metrics = self.evaluator.calculate_metrics(y_true, y_pred)

        # Calculate per-label metrics
        per_label_df = self.evaluator.calculate_per_label_metrics(
            y_true, y_pred, self.config['target_permissions']
        )

        # Store results
        result = {
            'model': model_name,
            'metrics': metrics,
            'per_label_metrics': per_label_df,
            'sample_size': len(test_descriptions),
            'few_shot': use_few_shot,
            'few_shot_examples_count': len(few_shot_examples) if few_shot_examples else 0
        }

        self.results[model_name] = result

        # Print results
        print(f"\n{'='*60}")
        print(f"Results for {model_name}:")
        print(f"{'='*60}")
        print(f"Hamming Loss: {metrics['hamming_loss']:.4f}")
        print(f"Exact Match Ratio: {metrics['exact_match_ratio']:.4f}")
        print(f"Jaccard Score (Macro): {metrics['jaccard_score']:.4f}")
        print(f"F1-Score (Macro): {metrics['f1_macro']:.4f}")
        print(f"F1-Score (Micro): {metrics['f1_micro']:.4f}")
        print(f"Precision (Macro): {metrics['precision_macro']:.4f}")
        print(f"Recall (Macro): {metrics['recall_macro']:.4f}")

        # Save results
        self.save_results(model_name, result, test_data, y_true, y_pred, raw_responses)

        return result

    def save_results(self, model_name: str, result: Dict[str, Any],
                    test_data: pd.DataFrame = None, y_true: List[List[int]] = None,
                    y_pred: List[List[int]] = None, raw_responses: List[str] = None):
        """
        Save results to JSON and CSV files, and predictions to Excel.

        Args:
            model_name: Name of the model
            result: Dictionary with results
            test_data: Test data DataFrame (optional, for saving predictions)
            y_true: True labels (optional, for saving predictions)
            y_pred: Predicted labels (optional, for saving predictions)
            raw_responses: Raw LLM response texts (optional, for saving to Excel)
        """
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        model_dir = os.path.join(self.config['results_dir'], model_name)
        os.makedirs(model_dir, exist_ok=True)

        # Create predictions directory
        predictions_dir = os.path.join(model_dir, "predictions")
        os.makedirs(predictions_dir, exist_ok=True)

        # Determine mode suffix for filenames
        mode_suffix = "zero_shot" if not result['few_shot'] else "few_shot"

        # Save JSON results
        json_file = os.path.join(model_dir, f"{model_name}_{mode_suffix}_multilabel_results_{timestamp}.json")
        with open(json_file, 'w') as f:
            json.dump({
                'model': model_name,
                'timestamp': timestamp,
                'metrics': result['metrics'],
                'sample_size': result['sample_size'],
                'few_shot': result['few_shot'],
                'few_shot_examples_count': result['few_shot_examples_count']
            }, f, indent=2)

        # Save per-label metrics to CSV
        csv_file = os.path.join(model_dir, f"{model_name}_{mode_suffix}_per_label_metrics_{timestamp}.csv")
        result['per_label_metrics'].to_csv(csv_file, index=False)

# Save predictions to Excel if provided
        if test_data is not None and y_true is not None and y_pred is not None:
            try:
                # Build predictions DataFrame
                pred_df = pd.DataFrame()

                # Add metadata columns if available
                metadata_cols = []
                for col in ['appId', 'description', 'name']:
                    if col in test_data.columns:
                        metadata_cols.append(col)
                        pred_df[col] = test_data[col].values

                # If no metadata columns, at least add description
                if not metadata_cols and 'description' in test_data.columns:
                    pred_df['description'] = test_data['description'].values

                # Add true labels for each permission
                for i, perm in enumerate(self.config['target_permissions']):
                    pred_df[f'true_{perm}'] = [y_true[j][i] for j in range(len(y_true))]

                # Add predicted labels for each permission
                for i, perm in enumerate(self.config['target_permissions']):
                    pred_df[f'predicted_{perm}'] = [y_pred[j][i] for j in range(len(y_pred))]

                # Add summary columns: number of true and predicted permissions
                pred_df['num_true_permissions'] = [sum(y_true[i]) for i in range(len(y_true))]
                pred_df['num_predicted_permissions'] = [sum(y_pred[i]) for i in range(len(y_pred))]
                pred_df['exact_match'] = [
                    1 if y_true[i] == y_pred[i] else 0
                    for i in range(len(y_true))
                ]

                # Add raw LLM response column if available
                if raw_responses is not None and len(raw_responses) == len(pred_df):
                    pred_df['llm_raw_response'] = raw_responses

                # Save predictions Excel (include zero-shot vs few-shot in filename)
                pred_path = os.path.join(predictions_dir, f"{model_name}_{mode_suffix}_predictions_{timestamp}.xlsx")
                pred_df.to_excel(pred_path, index=False)
                print(f"  📄 Predictions saved: {pred_path}")

            except Exception as e:
                print(f"  ⚠️ Could not save predictions Excel: {e}")
                import traceback
                traceback.print_exc()
        print(f"\nResults saved:")
        print(f"  JSON: {json_file}")
        print(f"  CSV: {csv_file}")

    def run_all_experiments(self, sample_size: int = None, use_few_shot: bool = True):
        """Run experiments for all models."""
        for model_name in self.config['llm_models'].keys():
            try:
                self.run_experiment(model_name, sample_size, use_few_shot)
            except Exception as e:
                print(f"Error running experiment for {model_name}: {e}")
                import traceback
                traceback.print_exc()
                continue

    def visualize_results(self):
        """Visualize results from all experiments."""
        if not self.results:
            print("No results to visualize. Run experiments first.")
            return

        # Extract metrics for comparison
        comparison_results = {
            model: result['metrics']
            for model, result in self.results.items()
        }

        self.evaluator.plot_metrics_comparison(comparison_results)

        return comparison_results

# Initialize pipeline
multilabel_pipeline = MultilabelClassificationPipeline(CONFIG)
print("Pipeline initialized and ready to use!")


Pipeline initialized and ready to use!


In [ ]:
# Example 1: Run experiment for a single model with few-shot learning
# Uncomment to run:
# results = multilabel_pipeline.run_experiment('mistral', sample_size=100, use_few_shot=True)

# Example 2: Run experiment without few-shot learning (zero-shot)
# results_flan_zero = multilabel_pipeline.run_experiment('flan-t5', sample_size=50, use_few_shot=False)

# Example 3: Run all experiments (this will take a long time)
# multilabel_pipeline.run_all_experiments(sample_size=100, use_few_shot=True)

# Example 4: Visualize results
# comparison = multilabel_pipeline.visualize_results()

print("Pipeline ready! Uncomment the examples above to run experiments.")
print("\nRecommended order for testing:")
print("1. FLAN-T5 (fastest, good baseline)")
print("2. Phi-3 (smaller model)")
print("3. Mistral (medium size)")
print("4. Llama 2 (largest, most capable)")
print("\nNote: Start with small sample sizes (10-50) for testing, then scale up.")


Pipeline ready! Uncomment the examples above to run experiments.

Recommended order for testing:
1. FLAN-T5 (fastest, good baseline)
2. Phi-3 (smaller model)
3. Mistral (medium size)
4. Llama 2 (largest, most capable)

Note: Start with small sample sizes (10-50) for testing, then scale up.


In [ ]:
results = multilabel_pipeline.run_experiment('llama2', sample_size=500, use_few_shot=False)


Running multilabel classification experiment with llama2
Few-shot learning: False
Loading data...
Data shape: (5348, 36)
Columns: 36
Preprocessing data...
Removed 0 rows with empty/short descriptions
Final data shape: (5348, 36)
Using sample of 500 records
Train set: 400 samples
Test set: 100 samples

Classifying 100 test samples...


Classifying with llama2...:   0%|          | 0/100 [00:00<?, ?it/s]

Loading llama2 (Llama 2 via HF + bnb 4-bit)...


tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

... (8 outputs left out)

Device set to use cuda:0
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


llama2 loaded successfully!


Classifying with llama2...: 100%|██████████| 100/100 [05:06<00:00,  3.07s/it]



Results for llama2:
Hamming Loss: 0.5010
Exact Match Ratio: 0.0000
Jaccard Score (Macro): 0.0000
F1-Score (Macro): 0.0000
F1-Score (Micro): 0.0000
Precision (Macro): 0.0000
Recall (Macro): 0.0000
  📄 Predictions saved: /content/drive/MyDrive/LLMs/multilabel_classification/results/llama2/predictions/llama2_zero_shot_predictions_20251120_073211.xlsx

Results saved:
  JSON: /content/drive/MyDrive/LLMs/multilabel_classification/results/llama2/llama2_zero_shot_multilabel_results_20251120_073211.json
  CSV: /content/drive/MyDrive/LLMs/multilabel_classification/results/llama2/llama2_zero_shot_per_label_metrics_20251120_073211.csv


In [ ]:
results = multilabel_pipeline.run_experiment('mistral', sample_size=500, use_few_shot=False)


Running multilabel classification experiment with mistral
Few-shot learning: False
Preprocessing data...
Removed 0 rows with empty/short descriptions
Final data shape: (5348, 36)
Using sample of 500 records
Train set: 400 samples
Test set: 100 samples

Classifying 100 test samples...


Classifying with mistral...:   0%|          | 0/100 [00:00<?, ?it/s]

Loading mistral...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (4 outputs left out)

Device set to use cuda:0
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


mistral loaded successfully!


Classifying with mistral...: 100%|██████████| 100/100 [12:45<00:00,  7.65s/it]



Results for mistral:
Hamming Loss: 0.5010
Exact Match Ratio: 0.0000
Jaccard Score (Macro): 0.0000
F1-Score (Macro): 0.0000
F1-Score (Micro): 0.0000
Precision (Macro): 0.0000
Recall (Macro): 0.0000
  📄 Predictions saved: /content/drive/MyDrive/LLMs/multilabel_classification/results/mistral/predictions/mistral_zero_shot_predictions_20251120_074458.xlsx

Results saved:
  JSON: /content/drive/MyDrive/LLMs/multilabel_classification/results/mistral/mistral_zero_shot_multilabel_results_20251120_074458.json
  CSV: /content/drive/MyDrive/LLMs/multilabel_classification/results/mistral/mistral_zero_shot_per_label_metrics_20251120_074458.csv


In [ ]:
# results = multilabel_pipeline.run_experiment('flan-t5', sample_size=500, use_few_shot=True)

In [ ]:
# results = multilabel_pipeline.run_experiment('phi-3', sample_size=500, use_few_shot=True)

## 8. Testing Individual Examples


In [ ]:
def debug_llm_response(model_name: str, description: str, use_few_shot: bool = True, show_prompt: bool = False):
    """
    Debug function to see what the LLM is actually outputting.

    Args:
        model_name: Name of the model to test
        description: App description to classify
        use_few_shot: Whether to use few-shot learning
        show_prompt: Whether to print the prompt being sent
    """
    print(f"{'='*60}")
    print(f"Debugging {model_name} response")
    print(f"{'='*60}")

    # Get classifier
    pipeline = MultilabelClassificationPipeline(CONFIG)
    classifier = pipeline.get_classifier(model_name)

    # Get few-shot examples if needed
    few_shot_examples = None
    if use_few_shot:
        data = pipeline.preprocessor.preprocess_data()
        train_data, _ = pipeline.preprocessor.split_data()
        few_shot_examples = pipeline.preprocessor.get_few_shot_examples(
            train_data,
            n_examples=CONFIG['few_shot_examples']
        )

    # Create prompt
    if hasattr(classifier, 'create_prompt'):
        prompt = classifier.create_prompt(description, few_shot_examples)
    else:
        prompt = classifier.create_prompt(description, few_shot_examples)

    if show_prompt:
        print(f"Prompt (first 500 chars):{prompt[:500]}...")

    # Get raw response
    if classifier.pipeline is None:
        classifier.load_model()

    try:
        if model_name.lower() == 'llama2':
            response = classifier.pipeline(
                prompt,
                max_new_tokens=200,
                do_sample=False,
                temperature=0.0,
                truncation=True,
                return_full_text=False
            )
        elif model_name.lower() == 'mistral':
            response = classifier.pipeline(
                prompt,
                max_new_tokens=200,
                do_sample=False,
                temperature=0.0,
                truncation=True,
                return_full_text=False
            )
        elif model_name.lower() == 'flan-t5':
            response = classifier.pipeline(
                prompt,
                max_new_tokens=200,
                do_sample=False,
                temperature=0.0,
                truncation=True
            )
        elif model_name.lower() == 'phi-3':
            response = classifier.pipeline(
                prompt,
                max_new_tokens=200,
                do_sample=False,
                temperature=0.0,
                truncation=True,
                return_full_text=False
            )
        else:
            response = classifier.pipeline(prompt, max_length=512, do_sample=False)

        if isinstance(response, list):
            response_text = response[0].get('generated_text', response[0].get('generated_text', ''))
        else:
            response_text = response.get('generated_text', '')

        print(f"Raw Response:{response_text}")
        print(f"Response length: {len(response_text)} characters")

        # Parse response
        predictions = classifier.parse_response(response_text)
        predicted_permissions = [
            CONFIG['target_permissions'][i]
            for i, pred in enumerate(predictions)
            if pred == 1
        ]

        print(f"Parsed Predictions ({sum(predictions)} permissions):")
        for perm in predicted_permissions:
            print(f"  - {perm}")

        if sum(predictions) == 0:
            print("⚠️ WARNING: No permissions were parsed from the response!")
            print("This suggests the parsing logic may need adjustment.")

        return response_text, predictions

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()
        return None, None
def test_single_example(model_name: str, description: str, use_few_shot: bool = True):
    """
    Test a single app description with a specific model.

    Args:
        model_name: Name of the model to use
        description: App description to classify
        use_few_shot: Whether to use few-shot learning
    """
    print(f"Testing {model_name} on a single example")
    print(f"Description: {description[:200]}...\n")

    # Get classifier
    pipeline = MultilabelClassificationPipeline(CONFIG)
    classifier = pipeline.get_classifier(model_name)

    # Get few-shot examples if needed
    few_shot_examples = None
    if use_few_shot:
        data = pipeline.preprocessor.preprocess_data()
        train_data, _ = pipeline.preprocessor.split_data()
        few_shot_examples = pipeline.preprocessor.get_few_shot_examples(
            train_data,
            n_examples=CONFIG['few_shot_examples']
        )
        print(f"Using {len(few_shot_examples)} few-shot examples\n")

    # Classify
    predictions = classifier.classify_permissions(description, few_shot_examples)

    # Get predicted permissions
    predicted_permissions = [
        CONFIG['target_permissions'][i]
        for i, pred in enumerate(predictions)
        if pred == 1
    ]

    print(f"Predicted permissions ({len(predicted_permissions)}):")
    for perm in predicted_permissions:
        print(f"  - {perm}")

    return predicted_permissions

# Example usage:
# predicted_permissions = test_single_example(
#     model_name='flan-t5',
#     description=str(processed_data.iloc[0]['description']),
#     use_few_shot=True
# )

# print(processed_data.iloc[0][CONFIG['target_permissions']])


In [ ]:
# debug_llm_response('flan-t5', description="This app requires access to your camera and photos", show_prompt=True)

Debugging flan-t5 response
Loading data...
Data shape: (5348, 36)
Columns: 36
Preprocessing data...
Removed 0 rows with empty/short descriptions
Final data shape: (5348, 36)
Train set: 4278 samples
Test set: 1070 samples
Prompt (first 500 chars):Task: Based on the app description, predict which permissions from the list below this app will request.



Here are some examples:

Example 1:
Description: Access Ambra Healths awardwinning medical image management suite right palm hand Users effortlessly view imaging anywhere scroll series utilize radiology tooling like zooming pinching rotating palm ha...
Permissions: Other : full network access

Example 2:
Description: Even better results less time PHALLOSAN forte app achieve goals even quic...
Loading flan-t5...


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


flan-t5 loaded successfully!
Raw Response:Other : run at startup, Camera : take pictures and videos, Wi-Fi connection information : view Wi-Fi connections, Location : precise location (GPS and network-based), Location : approximate location (network-based), Other : pair with Bluetooth devices, Microphone : record audio, Other : change your audio settings, Other : access Bluetooth settings, Calendar : read calendar events plus confidential information, Calendar : add or modify calendar events and send
Response length: 463 characters
Parsed Predictions (0 permissions):
⚠️ WARNING: No permissions were parsed from the response!
This suggests the parsing logic may need adjustment.


('Other : run at startup, Camera : take pictures and videos, Wi-Fi connection information : view Wi-Fi connections, Location : precise location (GPS and network-based), Location : approximate location (network-based), Other : pair with Bluetooth devices, Microphone : record audio, Other : change your audio settings, Other : access Bluetooth settings, Calendar : read calendar events plus confidential information, Calendar : add or modify calendar events and send',
 [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0])

## 10. Performance Improvement Tips

### Key Improvements Made:

1. **Better Prompt Design**:
   - Explicit output format instructions ("Permissions: [numbers]")
   - Clear step-by-step instructions
   - Better structured examples

2. **Improved Response Parsing**:
   - Multiple parsing strategies (structured format, numbers, permission names)
   - Validation of extracted numbers
   - Handling of edge cases ("none", "no permissions")

3. **Diverse Few-Shot Examples**:
   - Selects examples with varying numbers of permissions
   - Shows both permission numbers and names
   - Better representation of the task

4. **Error Handling**:
   - Validates predictions (prevents too many false positives)
   - Better error messages for debugging

### Additional Recommendations:

1. **Increase few-shot examples**: Try 5-7 examples instead of 3
2. **Use temperature=0**: Already implemented for deterministic results
3. **Post-process predictions**: Add validation based on permission co-occurrence patterns
4. **Try different models**: Some models may perform better on this task
5. **Fine-tune prompts**: Experiment with different prompt styles for your specific models


## 9. Summary

This notebook provides a comprehensive pipeline for **multilabel classification** of Android app permissions using different Large Language Models (LLMs) with **few-shot learning**.

### 🚀 **Key Features:**

#### **Core Pipeline:**
- **Few-shot Learning**: Uses example demonstrations to guide the model
- **Multilabel Classification**: Predicts multiple permissions per app simultaneously
- **Modular Design**: Easy to add new models or modify existing ones
- **4 LLM Support**: Llama 2, Mistral, FLAN-T5, and Phi-3
- **30 Target Permissions**: Comprehensive permission classification
- **Memory Efficient**: Uses quantization for large models
- **Reproducible**: Fixed random seeds and structured results

#### **Evaluation Metrics:**
- **Hamming Loss**: Average fraction of labels incorrectly predicted
- **Exact Match Ratio**: Fraction of samples with all labels correctly predicted
- **Jaccard Score**: Intersection over union of predicted and true labels
- **F1-Score (Macro/Micro)**: Macro and micro-averaged F1 scores
- **Per-Label Metrics**: Precision, Recall, F1 for each permission

### 📊 **Usage:**

```python
# Run single model experiment
results = multilabel_pipeline.run_experiment('flan-t5', sample_size=100, use_few_shot=True)

# Run all models
multilabel_pipeline.run_all_experiments(sample_size=100, use_few_shot=True)

# Visualize results
multilabel_pipeline.visualize_results()
```

### 📁 **Results Storage:**
Results are saved in `results/llm_multilabel_results/{model_name}/`:
- JSON files with overall metrics
- CSV files with per-label metrics

### 🎯 **Next Steps:**
1. Start with small sample sizes (10-50) for testing
2. Compare few-shot vs zero-shot performance
3. Experiment with different numbers of few-shot examples
4. Analyze per-label performance to identify challenging permissions
